<a href="https://colab.research.google.com/github/AureliaVDB/TickIt_Data_Pipeline/blob/main/TickIt_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install pyspark -q

In [5]:
from pyspark.sql import SparkSession, functions as F
import os, glob, shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os

In [3]:
spark = (SparkSession.builder
         .appName("TickIt_Medallion_Pipeline")
         .getOrCreate()
         )

In [6]:
from google.colab import drive

drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/TickIt_Project"

RAW = f"{BASE}/raw"  # csv files
BRONZE = f"{BASE}/bronze"  # storage for bronze layer
SILVER = f"{BASE}/silver"  # storage for silver layer
GOLD = f"{BASE}/gold"  # storage for gold layer

for path in [RAW, BRONZE, SILVER, GOLD]:
    os.makedirs(path, exist_ok=True)

Mounted at /content/drive


Bronze Layer

In [7]:
# list of all the source files
raw_files = []
for f in os.listdir(RAW):
    if f.endswith(".csv"):
        raw_files.append(f)

print(f"files in list: {raw_files}")

files in list: ['orders.csv', 'payments.csv', 'events.csv', 'event_categories.csv', 'ticket_types.csv', 'customers.csv', 'reviews.csv']


In [12]:
for file in raw_files:
  input_path = os.path.join(RAW, file)
  table_name = os.path.splitext(file)[0]
  output_path = os.path.join(BRONZE, table_name)

  df = spark.read.csv(input_path, header=True) # load the raw data
  df.write.mode("overwrite").parquet(output_path) # write to bronze layer

  print(f"{table_name.upper()}")
  print()
  print("Schema")
  df.printSchema()
  print(f"Row Count: {df.count()}")
  print()
  print("Sample Records")
  df.show(5)
  print()

ORDERS

Schema
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- ticket_type_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- event_date: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- promo_code: string (nullable = true)

Row Count: 52520

Sample Records
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|      order_id|     customer_id|event_id|ticket_type_id|         order_date|event_date|quantity|order_status|promo_code|
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|    ORD0039854|b2744cf668394fac| EVT0050|       TT00106|2022-08-15 10:00:00|2022-09-03|       2|   completed|      NULL|
|    ORD0002676|ad9181b36ee84121| EVT0092|       TT00194|2021-04-15 23:00:00|20